# Environment / Dependencies Setup
One command installs everything the lab needs:

In [1]:
!pip install -q "pymongo[srv,tls]==4.10.1" python-dotenv==1.0.1 certifi


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Step 1 — Connect to MongoDB
`load_dotenv()` reads `MONGODB_URI` from `.env`, and `certifi` supplies the CA bundle for a secure TLS connection.

In [8]:
import os
import certifi
from dotenv import load_dotenv
import pymongo

# Load the Atlas connection string from the .env file
load_dotenv()
uri = os.environ["MONGODB_URI"]

# Connect to the real MongoDB Atlas cluster
client = pymongo.MongoClient(uri, tlsCAFile=certifi.where())

# Access the database and collection
db = client["school_db"]
students = db["students"]

print("Connected to MongoDB Atlas")

Connected to MongoDB Atlas


### Step 2 — Insert the Starting Dataset
`students.drop()` clears the collection, then `insert_many()` loads the 25 starting records.

In [9]:
# Drop the collection first so re-running the notebook starts clean
students.drop()

# 25 student records — the starting point for write operations
student_records = [
    {"name": "Alice Johnson",  "student_id": "STU001", "course": "Computer Science", "grade": 95, "enrollment_date": "2024-09-01", "status": "active"},
    {"name": "Bob Smith",      "student_id": "STU002", "course": "Mathematics",      "grade": 78, "enrollment_date": "2024-09-01", "status": "active"},
    {"name": "Charlie Brown",  "student_id": "STU003", "course": "Physics",          "grade": 55, "enrollment_date": "2024-09-02", "status": "active"},
    {"name": "Diana Prince",   "student_id": "STU004", "course": "English",          "grade": 48, "enrollment_date": "2024-09-01", "status": "active"},
    {"name": "Eve Torres",     "student_id": "STU005", "course": "Biology",          "grade": 88, "enrollment_date": "2024-09-03", "status": "active"},
    {"name": "Frank Castle",   "student_id": "STU006", "course": "Mathematics",      "grade": 52, "enrollment_date": "2024-09-01", "status": "inactive"},
    {"name": "Grace Hopper",   "student_id": "STU007", "course": "Computer Science", "grade": 91, "enrollment_date": "2024-09-02", "status": "active"},
    {"name": "Hank Pym",       "student_id": "STU008", "course": "Physics",          "grade": 73, "enrollment_date": "2024-09-01", "status": "active"},
    {"name": "Ivan Petrov",    "student_id": "STU009", "course": "Computer Science", "grade": 84, "enrollment_date": "2024-09-03", "status": "active"},
    {"name": "Julia Child",    "student_id": "STU010", "course": "English",          "grade": 90, "enrollment_date": "2024-09-02", "status": "graduated"},
    {"name": "Karl Marx",      "student_id": "STU011", "course": "Mathematics",      "grade": 67, "enrollment_date": "2024-09-01", "status": "active"},
    {"name": "Laura Palmer",   "student_id": "STU012", "course": "Biology",          "grade": 45, "enrollment_date": "2024-09-02", "status": "active"},
    {"name": "Marco Polo",     "student_id": "STU013", "course": "Physics",          "grade": 82, "enrollment_date": "2024-09-03", "status": "active"},
    {"name": "Nina Simone",    "student_id": "STU014", "course": "English",          "grade": 76, "enrollment_date": "2024-09-01", "status": "active"},
    {"name": "Oscar Wilde",    "student_id": "STU015", "course": "Mathematics",      "grade": 93, "enrollment_date": "2024-09-02", "status": "active"},
    {"name": "Pia Zadora",     "student_id": "STU016", "course": "Biology",          "grade": 71, "enrollment_date": "2024-09-01", "status": "active"},
    {"name": "Quincy Adams",   "student_id": "STU017", "course": "Computer Science", "grade": 87, "enrollment_date": "2024-09-03", "status": "active"},
    {"name": "Rosa Parks",     "student_id": "STU018", "course": "English",          "grade": 58, "enrollment_date": "2024-09-02", "status": "inactive"},
    {"name": "Sam Wilson",     "student_id": "STU019", "course": "Physics",          "grade": 98, "enrollment_date": "2024-09-01", "status": "active"},
    {"name": "Tina Turner",    "student_id": "STU020", "course": "Mathematics",      "grade": 79, "enrollment_date": "2024-09-03", "status": "active"},
    {"name": "Uma Thurman",    "student_id": "STU021", "course": "Computer Science", "grade": 62, "enrollment_date": "2024-09-02", "status": "active"},
    {"name": "Vera Wang",      "student_id": "STU022", "course": "Biology",          "grade": 85, "enrollment_date": "2024-09-01", "status": "graduated"},
    {"name": "Walt Disney",    "student_id": "STU023", "course": "Physics",          "grade": 56, "enrollment_date": "2024-09-02", "status": "active"},
    {"name": "Xena Warrior",   "student_id": "STU024", "course": "English",          "grade": 94, "enrollment_date": "2024-09-03", "status": "active"},
    {"name": "Yusuf Islam",    "student_id": "STU025", "course": "Computer Science", "grade": 70, "enrollment_date": "2024-09-01", "status": "active"},
]

result = students.insert_many(student_records)
print(f"Inserted {len(result.inserted_ids)} student records.")

Inserted 25 student records.


### Step 3 — Single Update: Add a Scholarship Field
`update_one()` changes only the first matching document; `$set` adds the new `scholarship` field.

In [10]:
# update_one() modifies the first document that matches the filter
# $set creates the "scholarship" field if it doesn't exist
result = students.update_one(
    {"name": "Alice Johnson"},
    {"$set": {"scholarship": "Dean's Award"}}
)

print(f"Updated: Alice Johnson — added scholarship = 'Dean's Award'")

Updated: Alice Johnson — added scholarship = 'Dean's Award'


### Step 4 — Bulk Increment: Bump Failing Grades by 5
`update_many()` with `$inc` raises every failing grade by 5 atomically, avoiding a read-modify-write race.

In [11]:
# Capture who's failing BEFORE the update
failing_ids = [s["student_id"] for s in students.find(
    {"grade": {"$lt": 60}}, {"_id": 0, "student_id": 1}
)]

# update_many() modifies ALL documents that match the filter
# $inc adds 5 to the grade field (no need to read the current value first)
result = students.update_many(
    {"grade": {"$lt": 60}},
    {"$inc": {"grade": 5}}
)

print(f"Updated {result.modified_count} failing student(s) — grades incremented by 5")

# Look up the SAME students by their captured IDs, not by re-filtering on grade
updated_failing = students.find(
    {"student_id": {"$in": failing_ids}}, {"_id": 0}
)

print("\n--- Updated Failing Students ---")
for s in updated_failing:
    print(f"{s['name']:<20} | {s['course']:<12} | Grade: {s['grade']}")

Updated 6 failing student(s) — grades incremented by 5

--- Updated Failing Students ---
Charlie Brown        | Physics      | Grade: 60
Diana Prince         | English      | Grade: 53
Frank Castle         | Mathematics  | Grade: 57
Laura Palmer         | Biology      | Grade: 50
Rosa Parks           | English      | Grade: 63
Walt Disney          | Physics      | Grade: 61


### Step 5 — Upsert: Re-enroll a Withdrawn Student
`upsert=True` inserts a new document when the filter matches nothing, handling a re-enrollment in one call.

In [12]:
# upsert=True: if the filter matches, update it; if nothing matches, insert a new document
# Frank Castle (STU006) has status "inactive" — this changes it to "active"
result = students.update_one(
    {"student_id": "STU006"},
    {"$set": {"status": "active"}},
    upsert=True
)

if result.upserted_id:
    print(f"Upserted: new document inserted with _id = {result.upserted_id}")
else:
    print(f"Upserted: Frank Castle — status set to 'active'")

Upserted: Frank Castle — status set to 'active'


### Step 6 — Atomic Find-and-Modify: Bump a Grade
`find_one_and_update` updates one document and returns it in a single atomic operation.

In [ ]:
# find_one_and_update: finds one document, applies an update, and returns it
# By default it returns the document BEFORE the update — so we can see the old grade
old = students.find_one_and_update(
    {"student_id": "STU002"},
    {"$inc": {"grade": 5}},
    projection={"_id": 0, "name": 1, "grade": 1}
)

print(f"Updated: {old['name']} — grade {old['grade']} \u2192 {old['grade'] + 5}")

### Step 7 — Delete: Remove Graduated Students
`find_one_and_delete` removes one graduated student and returns the deleted document for logging.

In [13]:
# find_one_and_delete: finds one document, deletes it, and returns what it removed
graduated = students.find_one_and_delete(
    {"status": "graduated"},
    projection={"_id": 0, "name": 1, "status": 1}
)

if graduated:
    print(f"Deleted: {graduated['name']} (graduated)")

# delete_many() removes ALL remaining documents matching the filter
result = students.delete_many({"status": "graduated"})
print(f"Deleted {result.deleted_count} graduated student(s).")

Deleted: Julia Child (graduated)
Deleted 1 graduated student(s).


### Step 8 — Summary Report
Re-collects the metrics from each step into one formatted enrollment report.

In [14]:
# Count remaining students
total = students.count_documents({})

# Count how many students have the "scholarship" field (added in Step 3)
updated = students.count_documents({"scholarship": {"$exists": True}})

# Status distribution
pipeline = [
    {"$group": {"_id": "$status", "count": {"$sum": 1}}},
    {"$sort": {"_id": 1}}
]
status_counts = {doc["_id"]: doc["count"] for doc in students.aggregate(pipeline)}

# Students still failing after the grade bump
still_failing = list(students.find(
    {"grade": {"$lt": 60}},
    {"_id": 0}
).sort("grade", 1))

print("         ENROLLMENT TRACKER SUMMARY")
print(f"\nTotal students remaining: {total}")
print(f"Students with scholarship field: {updated}")

print("\n--- Status Distribution ---")
for status, count in sorted(status_counts.items()):
    print(f"  {status}: {count}")

print("\n--- Still Failing (Grade < 60) ---")
for s in still_failing:
    print(f"  {s['name']} ({s['course']}) — Grade: {s['grade']}")

         ENROLLMENT TRACKER SUMMARY

Total students remaining: 23
Students with scholarship field: 1

--- Status Distribution ---
  active: 22
  inactive: 1

--- Still Failing (Grade < 60) ---
  Laura Palmer (Biology) — Grade: 50
  Diana Prince (English) — Grade: 53
  Frank Castle (Mathematics) — Grade: 57
